## Setup

In [ ]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

In [ ]:
chapter_name = "setup"

from pyspark.sql import SparkSession

# Default memory will not be sufficient for the examples below, so we up it to 4GB per 
# executor
executor_memory = "4g"
executor_cores = 4
num_executors = 2


spark = SparkSession.builder \
        .appName(chapter_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
        .getOrCreate()

In [ ]:
# Create data and load into Iceberg tables for the code below
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

run(scaling_factor=1) # data will be 1GB on your disk
run_ddl(data_path=Path("./data"), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

In [ ]:
spark.sql("use prod.db") # this is the schema in which our data is

## Partitioning

In [ ]:
# FULL TABLE SCAN = All data is read into Spark Memory and then filtered
from pyspark.sql import functions as F

spark.read.table("lineitem").where(F.col("l_receiptdate") == "1992-01-05").explain()

In [ ]:
# create a partitioned table
spark\
.read\
.table("lineitem")\
.writeTo("lineitem_rdp")\
.partitionedBy("l_receiptdate")\
.createOrReplace()

![partitioned lineitem](images/part_folder.png)

In [ ]:
# PARTITION PRUNING = Only read the necessary partition
# No in memory filtering
spark\
.read\
.table("lineitem_rdp")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.explain()

In [ ]:
# Executing queries to see differences in the Spark UI at http://localhost:4040
spark\
.read\
.table("lineitem_rdp")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.show(2)

spark\
.read\
.table("lineitem")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.show(2)

![Non Partitioned v Partitioned](images/partition.png)

## Bucketing

In [ ]:
# NO BUCKETING GROUP BY = Exchange (movement of data between nodes in the cluster) 
# EXPENSIVE !!!
spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem
GROUP BY l_receiptdate
""").explain()

In [ ]:
# Create a bucketed table, with the bucketing based on the group by column
from pyspark.sql import functions as F

spark.read.table("lineitem")\
.writeTo("lineitem_bb_rd") \
.partitionedBy(  
    F.partitioning.bucket(4, "l_receiptdate")
).createOrReplace()

![Bucketed lineitem](images/bucket_folder.png)

In [ ]:
# BUCKETING = NO Exchange 
# settings to ensure Spark + Iceberg can leverage the underlying data layout
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem_bb_rd
GROUP BY l_receiptdate
""").explain()

## Sorting

In [ ]:
# Can only be noticed if we run the query
# NO SORTING = Large data read into Spark Memory
spark.sql("""
select *
from lineitem
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem", mode = 'overwrite')

In [ ]:
# Create Sorted Table
spark.read.table("lineitem")\
.sort("l_extendedprice")\
.coalesce(3)\
.writeTo("lineitem_sb_ep")\
.createOrReplace()

In [ ]:
# SORTING = Smaller data read into Spark Memory
spark.sql("""
select *
from lineitem_sb_ep
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem_0rd", mode = 'overwrite')

![Sorted v UnSorted Reads](images/sorting.png)